# 寄り引け（OPEN→CLOSE）戦略 分析ノートブック

J-Quants API データ（`data/jquantsapi/v2/`）を用いて、**前日売買代金 TOP20** を取引対象に、
当日の**寄り引け（翌日始値→終値）リターン**を予測対象とした分析を行う。

## データ
| ファイル | 内容 |
|---|---|
| `data/jquantsapi/v2/stock_prices_mod.parquet` | 価格データ（`O`/`C`/`AdjC` 等） |
| `data/jquantsapi/v2/stock_list_mod.parquet` | 銘柄リスト（信用取引区分 `Mrgn`） |
| `data/jquantsapi/v2/margin_alert.parquet` | 日々公表データ（売り禁 `RestrictedByJSF`） |

## 取引対象
- 前日の売買代金 TOP20。
- 銘柄リストの信用取引区分 `Mrgn` をマージして ffill。
- 日々公表データの売り禁 `RestrictedByJSF` をマージして ffill。

## 特徴量
- OPEN-CLOSE 騰落率 `C/O - 1`
- 前日比 `groupby(Code)[AdjC].pct_change(1)`
- 直近 5 日騰落率 `pct_change(5)`
- 直近 21 日騰落率 `pct_change(21)`

## 市況特徴量
- 前日の米国市況：S&P500（`^GSPC`）の CLOSE-CLOSE 騰落率をマージ。

## ターゲット
- 当日の寄り引けリターン
  ```python
  df["NextO"] = df.groupby("Code")["O"].shift(-1)
  df["NextC"] = df.groupby("Code")["C"].shift(-1)
  df["Target"] = df["NextC"] / df["NextO"] - 1
  ```

## 分析内容
- 直近 10 年を分析期間とする。
- 市況特徴量の正負の符号に応じて（米国市況が陽線／陰線）、各特徴量の符号に従って売買したときの
  バランスカーブをプロットする。


## 0. セットアップ

S&P500 は `yfinance` で `^GSPC` を取得する（要ネットワーク）。

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

# --- パス設定 ----------------------------------------------------------------
# このノートブックは notebooks/ 配下にある想定。リポジトリ直下を基準にする。
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(ROOT, "data")):
    ROOT = os.getcwd()  # ルート直下で実行している場合のフォールバック

DATA_DIR = os.path.join(ROOT, "data", "jquantsapi", "v2")
PRICE_PATH = os.path.join(DATA_DIR, "stock_prices_mod.parquet")
LIST_PATH = os.path.join(DATA_DIR, "stock_list_mod.parquet")
ALERT_PATH = os.path.join(DATA_DIR, "margin_alert.parquet")

# 結合キー（ユーザー確認済み: Date / Code）
DATE_COL = "Date"
CODE_COL = "Code"

print("DATA_DIR :", DATA_DIR)
for p in (PRICE_PATH, LIST_PATH, ALERT_PATH):
    print(("  OK " if os.path.exists(p) else "  NG "), p)


## 1. データ読み込み

価格データを読み込み、`Date`/`Code` を整える。`Date` は tz なしの `datetime64[ns]` に正規化する。

In [ ]:
price = pd.read_parquet(PRICE_PATH)
stock_list = pd.read_parquet(LIST_PATH)
alert = pd.read_parquet(ALERT_PATH)

print("price      :", price.shape)
print("stock_list :", stock_list.shape)
print("alert      :", alert.shape)
print("\nprice columns:", list(price.columns))
price.head()


In [ ]:
def normalize_date(df, col=DATE_COL):
    """Date 列を tz なし datetime64[ns] に正規化。"""
    s = pd.to_datetime(df[col])
    try:
        s = s.dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass
    df[col] = s.dt.normalize()
    return df

price = normalize_date(price)
df = price.copy()

# Code/Date で昇順ソート（groupby pct_change / ffill の前提）
df = df.sort_values([CODE_COL, DATE_COL]).reset_index(drop=True)
df.head()


## 2. 取引対象：前日の売買代金 TOP20

売買代金の列名を自動検出する。J-Quants 標準の `TurnoverValue` を優先し、無ければ
`終値 × 出来高` で近似する（売買代金列のデータが未提供のため、文脈から堅牢に判定）。

**前日の**売買代金で選別する点に注意。ある日 `t` の終値時点で判明している `t` 日の売買代金 TOP20 を、
翌営業日 `t+1` の寄り引けで取引する対象とする（＝翌日取引時点から見て「前日」の売買代金）。
リーク防止のため、ランクは各 `Date` 内で当日売買代金に基づいて付与し、ターゲットのみ `t+1` を参照する。

In [ ]:
def detect_turnover(df):
    """売買代金列を検出。無ければ 終値×出来高 で近似する。"""
    candidates = ["TurnoverValue", "Turnover", "TradingValue",
                  "DealValue", "Value", "TV", "Amount"]
    for c in candidates:
        if c in df.columns:
            print(f"売買代金列を検出: {c}")
            return df[c].astype(float)
    # フォールバック: 終値 × 出来高
    vol_candidates = ["V", "Volume", "Vol", "AdjV", "AdjVolume"]
    close_col = "C" if "C" in df.columns else ("Close" if "Close" in df.columns else None)
    vol_col = next((v for v in vol_candidates if v in df.columns), None)
    if close_col and vol_col:
        print(f"売買代金列が無いため {close_col} x {vol_col} で近似")
        return df[close_col].astype(float) * df[vol_col].astype(float)
    raise KeyError("売買代金を構成できる列が見つかりません: " + str(list(df.columns)))

df["Turnover"] = detect_turnover(df)

# 各 Date 内で売買代金ランク（1 が最大）。当日売買代金で判定 = 翌日取引から見て「前日」。
TOP_N = 20
df["TurnoverRank"] = df.groupby(DATE_COL)["Turnover"].rank(ascending=False, method="first")
df["InUniverse"] = df["TurnoverRank"] <= TOP_N

print("ユニバース該当行数:", int(df["InUniverse"].sum()))
print("1日あたり平均銘柄数:", df.groupby(DATE_COL)["InUniverse"].sum().mean())


## 3. 信用取引区分・売り禁のマージ（ffill）

- 銘柄リストの信用取引区分 `Mrgn` をマージして ffill。
- 日々公表データの売り禁 `RestrictedByJSF` をマージして ffill。

ffill は **全パネル（全銘柄×全日付）** に対して銘柄ごとに行う（ユニバース抽出は後段）。
日々公表データは売り禁発生日のみ疎に存在するため、銘柄ごとに前方補完して状態を継続させる。

In [ ]:
# --- Mrgn（信用取引区分）のマージ ---------------------------------------------
list_cols = list(stock_list.columns)
print("stock_list columns:", list_cols)

if "Mrgn" not in stock_list.columns:
    raise KeyError("stock_list に Mrgn 列がありません: " + str(list_cols))

if DATE_COL in stock_list.columns:
    # 日付つき（時系列で変化しうる）→ Date/Code でマージ後 ffill
    sl = normalize_date(stock_list.copy())
    sl = sl[[DATE_COL, CODE_COL, "Mrgn"]].drop_duplicates()
    df = df.merge(sl, on=[DATE_COL, CODE_COL], how="left")
else:
    # 静的（銘柄ごとに1つ）→ Code でマージ
    sl = stock_list[[CODE_COL, "Mrgn"]].drop_duplicates(subset=[CODE_COL])
    df = df.merge(sl, on=CODE_COL, how="left")

df = df.sort_values([CODE_COL, DATE_COL]).reset_index(drop=True)
df["Mrgn"] = df.groupby(CODE_COL)["Mrgn"].ffill()
print("Mrgn 欠損率:", df["Mrgn"].isna().mean())


In [ ]:
# --- RestrictedByJSF（売り禁）のマージ ----------------------------------------
alert_cols = list(alert.columns)
print("alert columns:", alert_cols)

if "RestrictedByJSF" not in alert.columns:
    raise KeyError("margin_alert に RestrictedByJSF 列がありません: " + str(alert_cols))

al = normalize_date(alert.copy())
al = al[[DATE_COL, CODE_COL, "RestrictedByJSF"]].drop_duplicates(subset=[DATE_COL, CODE_COL])

df = df.merge(al, on=[DATE_COL, CODE_COL], how="left")
df = df.sort_values([CODE_COL, DATE_COL]).reset_index(drop=True)
# 売り禁状態を銘柄ごとに前方補完（発生日以降を継続）。初期欠損は「規制なし」とみなす。
df["RestrictedByJSF"] = df.groupby(CODE_COL)["RestrictedByJSF"].ffill()
print("RestrictedByJSF 値の分布:")
print(df["RestrictedByJSF"].value_counts(dropna=False))


## 4. 特徴量

- `OC`  : OPEN-CLOSE 騰落率 `C / O - 1`
- `ret1`: 前日比 `pct_change(1)`（`AdjC`）
- `ret5`: 直近 5 日騰落率 `pct_change(5)`
- `ret21`: 直近 21 日騰落率 `pct_change(21)`

`pct_change` は銘柄ごと（`groupby(Code)`）に、Date 昇順で計算する。

In [ ]:
# OPEN-CLOSE 騰落率
df["OC"] = df["C"] / df["O"] - 1

# 各種リターン（AdjC ベース、銘柄ごと）
df["ret1"] = df.groupby(CODE_COL)["AdjC"].pct_change(1)
df["ret5"] = df.groupby(CODE_COL)["AdjC"].pct_change(5)
df["ret21"] = df.groupby(CODE_COL)["AdjC"].pct_change(21)

FEATURES = ["OC", "ret1", "ret5", "ret21"]
df[FEATURES].describe()


## 5. 市況特徴量：前日の米国市況（S&P500 CLOSE-CLOSE）

`yfinance` で `^GSPC` を取得し、CLOSE-CLOSE 騰落率を計算する。

日本市場の取引日 `d` の意思決定時点（前日終値）で判明しているのは、**その朝までにクローズした
直近の米国セッション**＝米国カレンダーで `d-1` のクローズである。したがって `merge_asof` の
`direction="backward", allow_exact_matches=False` で、各日本取引日に対し**厳密に前の**米国営業日の
リターンを割り当てる（リーク防止）。

In [ ]:
import yfinance as yf

start = (df[DATE_COL].min() - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
end = (df[DATE_COL].max() + pd.Timedelta(days=2)).strftime("%Y-%m-%d")

sp = yf.download("^GSPC", start=start, end=end, auto_adjust=False, progress=False)
# yfinance のバージョン差（MultiIndex 列）に対応
if isinstance(sp.columns, pd.MultiIndex):
    sp.columns = sp.columns.get_level_values(0)
sp = sp[["Close"]].rename(columns={"Close": "SP500_Close"}).reset_index()
sp = sp.rename(columns={"Date": DATE_COL})
sp = normalize_date(sp)
sp = sp.sort_values(DATE_COL).reset_index(drop=True)

# CLOSE-CLOSE 騰落率
sp["SP500_cc"] = sp["SP500_Close"].pct_change(1)
sp = sp.dropna(subset=["SP500_cc"]).reset_index(drop=True)
sp.tail()


In [ ]:
# 各日本取引日に「厳密に前の米国営業日」の CLOSE-CLOSE リターンを割り当てる
jp_dates = pd.DataFrame({DATE_COL: np.sort(df[DATE_COL].unique())})
mkt = pd.merge_asof(
    jp_dates,
    sp[[DATE_COL, "SP500_cc"]],
    on=DATE_COL,
    direction="backward",
    allow_exact_matches=False,
)
df = df.merge(mkt, on=DATE_COL, how="left")
print("SP500_cc 欠損率:", df["SP500_cc"].isna().mean())
df[[DATE_COL, CODE_COL, "SP500_cc"]].dropna().head()


## 6. ターゲット：翌日の寄り引けリターン

```python
df["NextO"] = df.groupby("Code")["O"].shift(-1)
df["NextC"] = df.groupby("Code")["C"].shift(-1)
df["Target"] = df["NextC"] / df["NextO"] - 1
```

In [ ]:
df["NextO"] = df.groupby(CODE_COL)["O"].shift(-1)
df["NextC"] = df.groupby(CODE_COL)["C"].shift(-1)
df["Target"] = df["NextC"] / df["NextO"] - 1
df["Target"].describe()


## 7. 分析期間：直近 10 年

データ最終日から遡って 10 年を分析対象とする。

In [ ]:
last_date = df[DATE_COL].max()
start_date = last_date - pd.DateOffset(years=10)
print("分析期間:", start_date.date(), "〜", last_date.date())

# ユニバース（前日売買代金 TOP20）かつ直近10年、ターゲット・市況が揃う行
mask = (
    df["InUniverse"]
    & (df[DATE_COL] >= start_date)
    & df["Target"].notna()
    & df["SP500_cc"].notna()
)
panel = df.loc[mask].copy()
print("分析対象行数:", len(panel))
print("対象営業日数:", panel[DATE_COL].nunique())
panel[[DATE_COL, CODE_COL, "Mrgn", "RestrictedByJSF"] + FEATURES + ["SP500_cc", "Target"]].head()


## 8. 分析：市況の符号 × 各特徴量の符号で売買したバランスカーブ

**売買ルール**
- 各特徴量について、ポジション = `sign(特徴量)`（プラスならロング、マイナスならショート）。
- 日次リターン = ユニバース内の `sign(特徴量) × Target` の等加重平均。

**市況の符号で分割**
- 前日の米国市況 `SP500_cc` が **プラスの日（米国陽線）** と **マイナスの日（米国陰線）** に分けて、
  それぞれバランスカーブ（資産曲線）を描く。
- これにより、「米国が上げた翌日／下げた翌日」で各特徴量の符号売買の挙動がどう変わるかを比較する。

バランスカーブは日次リターンの累積（複利、`(1+r).cumprod()`）で表現する。

In [ ]:
def daily_returns_for_feature(data, feat):
    """各日付について sign(feat)*Target の等加重平均（日次戦略リターン）を返す。"""
    d = data[[DATE_COL, feat, "Target"]].dropna()
    pos = np.sign(d[feat])
    d = d.assign(pnl=pos * d["Target"])
    return d.groupby(DATE_COL)["pnl"].mean()


# 市況レジームでパネルを分割
panel = panel.sort_values([DATE_COL, CODE_COL])
up_days = panel[panel["SP500_cc"] > 0]      # 米国陽線の翌日
down_days = panel[panel["SP500_cc"] < 0]    # 米国陰線の翌日

print("米国陽線レジーム 営業日数:", up_days[DATE_COL].nunique())
print("米国陰線レジーム 営業日数:", down_days[DATE_COL].nunique())


def balance_curve(daily):
    """日次リターン series -> バランスカーブ(初期値1.0, 複利)。"""
    daily = daily.sort_index()
    return (1.0 + daily).cumprod()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
axes = axes.ravel()

summary_rows = []
for ax, feat in zip(axes, FEATURES):
    for label, subset, color in [
        ("US up (米国陽線の翌日)", up_days, "tab:blue"),
        ("US down (米国陰線の翌日)", down_days, "tab:red"),
    ]:
        daily = daily_returns_for_feature(subset, feat)
        if daily.empty:
            continue
        curve = balance_curve(daily)
        ax.plot(curve.index, curve.values, label=label, color=color)

        total = curve.iloc[-1] - 1.0
        ann = (1.0 + daily.mean()) ** 252 - 1.0
        sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else np.nan
        summary_rows.append({
            "feature": feat, "regime": label,
            "total_return": total, "ann_return": ann, "sharpe": sharpe,
            "n_days": daily.shape[0],
        })

    ax.axhline(1.0, color="gray", lw=0.8, ls="--")
    ax.set_title(f"feature = {feat}  (position = sign({feat}))")
    ax.set_ylabel("balance (start=1.0)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle("市況の符号で分割した、各特徴量の符号売買のバランスカーブ（直近10年・前日売買代金TOP20）",
             fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


In [ ]:
# サマリー（参考指標）
summary = pd.DataFrame(summary_rows)
summary["total_return"] = (summary["total_return"] * 100).round(2)
summary["ann_return"] = (summary["ann_return"] * 100).round(2)
summary["sharpe"] = summary["sharpe"].round(2)
summary = summary.rename(columns={"total_return": "total_return(%)", "ann_return": "ann_return(%)"})
summary


### 補足
- ポジションは符号のみ（±1）の等加重。コスト・売買単位・流動性制約は未考慮の理論値。
- `RestrictedByJSF`（売り禁）や `Mrgn`（信用区分）はパネルに保持済み。ショート不可銘柄を除外したい場合は、
  例えば `panel = panel[panel["RestrictedByJSF"] != <売り禁フラグ>]` のようにフィルタして再実行する。
- 市況レジーム（陽線/陰線）でバランスカーブが乖離するなら、米国市況に条件づけた符号売買に妙味がある可能性を示す。
- S&P500 は `yfinance` 取得のため、ネットワーク制限下では別ソース（parquet 等）への差し替えを検討する。
